In [ ]:
df = courier_data["shipments"]
date_range = courier_data["date_range"]

st.subheader("Courier & Logistics Performance")
st.caption(
    f"Shipment-level data from Shiprocket covering {date_range[0]} to {date_range[1]} "
    f"({len(df):,} shipments). This is independent of the Shopify order export used elsewhere "
    "in the dashboard, so totals won't match exactly -- it's one row per shipment leg (a multi-SKU "
    "order can have multiple rows) and only orders actually handed to a courier appear here."
)

total = len(df)
delivered = df["Is_Delivered"].sum()
rto = df["Is_RTO"].sum()
cancelled = df["Is_Cancelled"].sum()

c1, c2, c3, c4, c5 = st.columns(5)
with c1:
    kpi_card("Total Shipments", f"{total:,}")
with c2:
    kpi_card("Delivered", f"{delivered/total*100:.1f}%")
with c3:
    kpi_card("RTO Rate", f"{rto/total*100:.1f}%", positive=False)
with c4:
    kpi_card("Cancelled", f"{cancelled/total*100:.1f}%", positive=False)
with c5:
    kpi_card("Avg Delivery Time", f"{df['Delivery_Days'].mean():.1f} days")

st.caption(
    "EDD (estimated delivery date) is corrupted in nearly every row of these exports -- a Shiprocket "
    "export-formatting issue, not something fixable from this data -- so true SLA adherence (actual "
    "vs. promised delivery date) can't be computed. 'Avg Delivery Time' above is order-placed to "
    "order-delivered instead."
)

In [ ]:
st.subheader("Courier Performance")
courier_stats = df.groupby("Courier").agg(
    Shipments=("Order ID", "count"),
    Delivered_Pct=("Is_Delivered", "mean"),
    RTO_Pct=("Is_RTO", "mean"),
    Avg_Delivery_Days=("Delivery_Days", "mean"),
)
courier_stats = courier_stats[courier_stats["Shipments"] >= 100]
courier_stats["Delivered_Pct"] *= 100
courier_stats["RTO_Pct"] *= 100
courier_stats = courier_stats.round(2).sort_values("Shipments", ascending=False)

col1, col2 = st.columns(2)
with col1:
    cs = courier_stats.reset_index().sort_values("Delivered_Pct")
    fig = px.bar(cs, x="Delivered_Pct", y="Courier", orientation="h",
                  title="Delivery Rate by Courier (%)", color_discrete_sequence=[SUCCESS])
    st.plotly_chart(fig, width="stretch")
with col2:
    cs2 = courier_stats.reset_index().sort_values("RTO_Pct", ascending=False)
    fig = px.bar(cs2, x="RTO_Pct", y="Courier", orientation="h",
                  title="RTO Rate by Courier (%)", color_discrete_sequence=[DANGER])
    st.plotly_chart(fig, width="stretch")

st.dataframe(courier_stats, width="stretch")
st.caption(
    "Couriers with under 100 shipments in this data are excluded (too few to be meaningful). "
    "Use this to shift volume away from chronically high-RTO/slow couriers toward the better performers."
)

In [ ]:
st.subheader("Performance by Delivery Zone")
zone_stats = df.groupby("Zone Label").agg(
    Shipments=("Order ID", "count"),
    Delivered_Pct=("Is_Delivered", "mean"),
    RTO_Pct=("Is_RTO", "mean"),
    Avg_Delivery_Days=("Delivery_Days", "mean"),
)
zone_stats["Delivered_Pct"] *= 100
zone_stats["RTO_Pct"] *= 100
zone_stats = zone_stats.round(2).sort_values("Shipments", ascending=False)

col1, col2 = st.columns(2)
with col1:
    fig = px.bar(zone_stats.reset_index(), x="Zone Label", y="Shipments",
                  title="Shipment Volume by Zone", color_discrete_sequence=[ACCENT1])
    st.plotly_chart(fig, width="stretch")
with col2:
    zs = zone_stats.reset_index()
    fig = go.Figure()
    fig.add_bar(x=zs["Zone Label"], y=zs["RTO_Pct"], name="RTO %", marker_color=DANGER)
    fig.add_bar(x=zs["Zone Label"], y=zs["Avg_Delivery_Days"], name="Avg Delivery Days",
                 marker_color=ACCENT2, yaxis="y2")
    fig.update_layout(title="RTO Rate & Avg Delivery Days by Zone",
                        yaxis=dict(title="RTO %"),
                        yaxis2=dict(title="Avg Days", overlaying="y", side="right"))
    st.plotly_chart(fig, width="stretch")

st.caption(
    "Zones are Shiprocket's distance-based pricing tiers (A = same city/local, B = same state, "
    "C = metro-to-metro, D = rest of India, E = remote/North-East/J&K). Most volume sits in Zone D, "
    "so that's the zone worth optimizing first."
)

In [ ]:
st.subheader("Delivery Attempts vs. Outcome")
st.caption(
    "Each failed delivery attempt (NDR -- Non-Delivery Report) sharply raises the odds of an "
    "eventual RTO. Use this to decide how many retry attempts are worth making before writing off "
    "a shipment."
)
attempt_outcome = (
    df[df["Attempt Count"].notna() & (df["Attempt Count"] <= 5)]
    .groupby("Attempt Count")["Outcome"].value_counts(normalize=True).unstack().fillna(0) * 100
)
fig = px.bar(attempt_outcome.reset_index(), x="Attempt Count", y=attempt_outcome.columns.tolist(),
              title="Outcome Mix by Number of Delivery Attempts (%)", barmode="stack",
              color_discrete_sequence=PALETTE)
fig.update_layout(yaxis_title="% of shipments", legend_title="Outcome")
st.plotly_chart(fig, width="stretch")

st.subheader("Top Reasons Deliveries Fail")
ndr = df["Latest NDR Reason"].dropna().value_counts().head(12).reset_index()
ndr.columns = ["Reason", "Count"]
fig = px.bar(ndr.sort_values("Count"), x="Count", y="Reason", orientation="h",
              title="Top NDR (Failed Delivery Attempt) Reasons", color_discrete_sequence=[WARNING])
st.plotly_chart(fig, width="stretch")

In [ ]:
st.subheader("Does Shiprocket's Risk Score Actually Predict RTO?")
risk_col = "Order Risk"
if df[risk_col].notna().any():
    risk_rto = df[df[risk_col].notna()].groupby(risk_col)["Is_RTO"].agg(["mean", "count"])
    risk_rto.columns = ["RTO_Rate", "Shipments"]
    risk_rto["RTO_Rate"] *= 100
    order = ["low", "high", "very-high"]
    risk_rto = risk_rto.reindex([o for o in order if o in risk_rto.index])
    fig = px.bar(risk_rto.reset_index(), x=risk_col, y="RTO_Rate",
                  title="RTO Rate by Shiprocket Order-Risk Tier (%)", color_discrete_sequence=[DANGER],
                  text="Shipments")
    st.plotly_chart(fig, width="stretch")
    st.caption(
        "Risk tiers are Shiprocket's own pre-shipment fraud/RTO score (based on address quality, "
        "order history, etc). The gap between tiers tells you whether it's worth acting on -- e.g. "
        "calling to confirm 'very-high' risk COD orders before shipping."
    )
else:
    st.info("Order Risk score not present in this batch of exports.")

st.subheader("Top Reasons for RTO")
rto_reason = df["RTO Reason"].dropna().value_counts().head(10)
if not rto_reason.empty:
    st.dataframe(
        rto_reason.reset_index().rename(columns={"index": "Reason", "RTO Reason": "Count"}),
        width="stretch",
    )
else:
    st.caption(
        "'RTO Reason' is populated for too few shipments in this data to be useful directly -- see "
        "'Latest NDR Reason' above instead, which captures the same failed-delivery causes far more completely."
    )

In [ ]:
st.subheader("COD Remittance Timing")
lag = df["COD_Remittance_Lag_Days"].dropna()
if not lag.empty:
    c1, c2, c3 = st.columns(3)
    with c1:
        kpi_card("Avg Remittance Lag", f"{lag.mean():.1f} days")
    with c2:
        kpi_card("Median Remittance Lag", f"{lag.median():.1f} days")
    with c3:
        kpi_card("Slowest 5%", f"{lag.quantile(0.95):.1f} days", positive=False)
    fig = px.histogram(lag, nbins=40, title="Days from Delivery to COD Remittance",
                         color_discrete_sequence=[ACCENT1])
    fig.update_layout(showlegend=False, xaxis_title="Days", yaxis_title="Shipments")
    st.plotly_chart(fig, width="stretch")
    st.caption(
        "How long cash sits with the courier after a COD delivery before it's remitted to you -- "
        "this is working capital tied up in transit."
    )
else:
    st.info("Not enough COD remittance date data to analyze.")

In [ ]:
st.subheader("Delivery & RTO Rate Trend")
monthly = df.groupby("Order Month").agg(
    Shipments=("Order ID", "count"),
    Delivered_Pct=("Is_Delivered", "mean"),
    RTO_Pct=("Is_RTO", "mean"),
).reset_index()
monthly["Order Month"] = monthly["Order Month"].astype(str)
monthly["Delivered_Pct"] *= 100
monthly["RTO_Pct"] *= 100

fig = go.Figure()
fig.add_scatter(x=monthly["Order Month"], y=monthly["Delivered_Pct"], name="Delivered %",
                  line=dict(color=SUCCESS))
fig.add_scatter(x=monthly["Order Month"], y=monthly["RTO_Pct"], name="RTO %", line=dict(color=DANGER))
fig.update_layout(title="Monthly Delivered % vs RTO % (by order month)", yaxis_title="%")
st.plotly_chart(fig, width="stretch")
st.caption(
    "The most recent 1-2 months will look artificially low on 'Delivered %' since many of those "
    "shipments are still in transit and haven't resolved yet."
)

In [ ]:
st.subheader("RTO Rate by State")
state_stats = df.groupby("Address State").agg(
    Shipments=("Order ID", "count"),
    RTO_Pct=("Is_RTO", "mean"),
    Delivered_Pct=("Is_Delivered", "mean"),
)
state_stats = state_stats[state_stats["Shipments"] >= 200]
state_stats["RTO_Pct"] *= 100
state_stats["Delivered_Pct"] *= 100
state_stats = state_stats.round(2).sort_values("RTO_Pct", ascending=False)

fig = px.bar(state_stats.reset_index().head(15), x="RTO_Pct", y="Address State", orientation="h",
              title="Highest RTO-Rate States (min. 200 shipments)", color_discrete_sequence=[DANGER])
fig.update_layout(yaxis=dict(autorange="reversed"))
st.plotly_chart(fig, width="stretch")
st.dataframe(state_stats, width="stretch")
st.caption("States with under 200 shipments in this data are excluded (too few to be meaningful).")